# Cleaning & Preprocessing — Craigslist Used Cars

**Objectif** : produire un dataset propre, encodé et splité (train/test), prêt à être consommé par `src/data.py::load_dataset_split()`.

## Pipeline

1. Chargement des deux sources : Craigslist (`vehicles.csv`) + prix neufs MSRP (`new_cars_msrp.csv`)
2. Nettoyage initial : drop colonnes inutiles, filtres basiques sur `price`, `year`, `odometer`
3. **Détection d'outliers via MSRP** : un véhicule d'occasion dont le prix dépasse le MSRP max d'un véhicule neuf de même marque/année est aberrant
4. Gestion des valeurs manquantes (par colonne)
5. Feature engineering (`age`, `posting_month`, `is_premium_brand`)
6. Train/test split (80/20)
7. Encoding (target encoding sur `manufacturer`, frequency encoding sur `model`, one-hot sur les autres) + scaling
8. Sauvegarde dans `data/vehicles_processed.parquet`

## 0. Imports & paths

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR = Path("..") / "data"
USED_PATH = DATA_DIR / "vehicles.csv"
MSRP_PATH = DATA_DIR / "new_cars_msrp.csv"
OUTPUT_PATH = DATA_DIR / "vehicles_processed.parquet"

RANDOM_STATE = 42
SCRAPE_YEAR = 2021  # année de scraping du dataset Craigslist

for p in (USED_PATH, MSRP_PATH):
    assert p.exists(), f"Missing file : {p}"

## 1. Chargement des datasets

In [2]:
# Craigslist : on dropne dès la lecture les colonnes lourdes inutiles pour la modélisation.
# /!\ description est conservée : on l'utilise pour extraire des features texte (1st owner, service history, etc.)
# puis elle sera supprimée à la fin du feature engineering.
COLS_TO_DROP = ["id", "url", "region_url", "image_url", "VIN"]

df = pd.read_csv(USED_PATH, parse_dates=["posting_date"])
df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])
print(f"Used cars (Craigslist) : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df.head(2)

Used cars (Craigslist) : 426,880 lignes x 20 colonnes


,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,county,state,lat,long,posting_date
0,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaT
1,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaT


In [3]:
msrp = pd.read_csv(MSRP_PATH)
print(f"MSRP (new cars)        : {msrp.shape[0]:,} lignes x {msrp.shape[1]} colonnes")
print(f"Plage d'années couverte : {msrp['Year'].min()} → {msrp['Year'].max()}")
msrp.head(2)

MSRP (new cars)        : 11,914 lignes x 16 colonnes
Plage d'années couverte : 1990 → 2017


,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650


## 2. Nettoyage initial du dataset Craigslist

On applique des filtres de bon sens **avant** la détection d'outliers via MSRP, pour éliminer les données clairement aberrantes (prix nuls, années impossibles, kilométrages absurdes).

In [4]:
n0 = len(df)

# 2.1 — Drop les lignes sans prix ou avec prix aberrant
df = df[df["price"].between(500, 250_000)]

# 2.2 — Drop les lignes sans année ou année impossible
df = df.dropna(subset=["year"])
df = df[df["year"].between(1980, SCRAPE_YEAR + 1)]
df["year"] = df["year"].astype("int16")

# 2.3 — Drop les lignes sans manufacturer (impossible à matcher avec MSRP)
df = df.dropna(subset=["manufacturer"])

# 2.4 — Filtre kilométrage : garder NaN (gérés plus tard) et valeurs réalistes
df = df[(df["odometer"].isna()) | (df["odometer"].between(0, 500_000))]

print(f"Après nettoyage initial : {len(df):,} lignes ({100*len(df)/n0:.1f} % conservées sur {n0:,})")

Après nettoyage initial : 362,192 lignes (84.8 % conservées sur 426,880)


## 3. Détection d'outliers via le dataset MSRP

**Règle (stricte)** : un véhicule d'occasion ne devrait **jamais** être affiché à un prix supérieur au MSRP (Manufacturer's Suggested Retail Price) de la même marque la même année.

→ Si `price_occasion > MSRP_max(marque, année)`, la ligne est considérée comme aberrante et supprimée.

On utilise `MSRP_max` (et non `MSRP_mean`) sur les couples `(marque, année)` pour ne pas pénaliser les variantes haut de gamme d'un modèle (ex. : la version Sport ou la finition Premium est plus chère que la version de base, mais reste une voiture neuve légitime).

In [5]:
# 3.1 — Normaliser les noms de marque dans les deux datasets pour permettre le matching
def norm_make(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

msrp["_make_norm"] = norm_make(msrp["Make"])
df["_make_norm"] = norm_make(df["manufacturer"])

# Aperçu de l'overlap
makes_used = set(df["_make_norm"].unique())
makes_msrp = set(msrp["_make_norm"].unique())
print(f"Marques uniques (Craigslist) : {len(makes_used)}")
print(f"Marques uniques (MSRP)       : {len(makes_msrp)}")
print(f"Marques communes              : {len(makes_used & makes_msrp)}")

Marques uniques (Craigslist) : 42
Marques uniques (MSRP)       : 48
Marques communes              : 30


In [6]:
# 3.2 — Construire la table de référence : pour chaque (make, year), MSRP max
msrp_lookup = (
    msrp.groupby(["_make_norm", "Year"], as_index=False)["MSRP"]
    .max()
    .rename(columns={"_make_norm": "_make_norm", "Year": "year", "MSRP": "msrp_max"})
)
msrp_lookup["year"] = msrp_lookup["year"].astype("int16")
print(f"Lookup MSRP : {len(msrp_lookup):,} couples (marque, année)")
msrp_lookup.head()

Lookup MSRP : 735 couples (marque, année)


,_make_norm,year,msrp_max
0,acura,1992,2000
1,acura,1993,2060
2,acura,1994,2384
3,acura,1995,2506
4,acura,1997,2488


In [7]:
# 3.3 — Joindre la lookup au dataset Craigslist
df = df.merge(msrp_lookup, on=["_make_norm", "year"], how="left")

# Combien de lignes ont une référence MSRP ?
covered = df["msrp_max"].notna().sum()
print(f"Lignes avec référence MSRP : {covered:,} ({100*covered/len(df):.1f} %)")
print("→ Les autres ne pourront pas être filtrées par cette règle (years/marques absentes du MSRP).")

Lignes avec référence MSRP : 232,704 (64.2 %)
→ Les autres ne pourront pas être filtrées par cette règle (years/marques absentes du MSRP).


In [8]:
# 3.4 — Détecter les outliers : règle stricte price > MSRP_max
df["_is_outlier_msrp"] = (
    df["msrp_max"].notna() & (df["price"] > df["msrp_max"])
)
n_outliers = df["_is_outlier_msrp"].sum()
print(f"Outliers détectés (price > MSRP_max) : {n_outliers:,} ({100*n_outliers/len(df):.2f} %)")

# Quelques exemples d'outliers
df.loc[df["_is_outlier_msrp"], ["manufacturer", "model", "year", "price", "msrp_max"]].head(10)

Outliers détectés (price > MSRP_max) : 8,125 (2.24 %)


,manufacturer,model,year,price,msrp_max
153,ford,f150,1998,5900,4059.0
154,dodge,charger limousine,2012,34995,25995.0
166,ford,f150,1997,3750,3550.0
188,chrysler,300m,1999,3300,2202.0
193,chevrolet,tahoe,1996,3500,2000.0
213,chevrolet,corvette,2000,13500,3465.0
269,buick,lesabre,1997,2300,2000.0
328,ford,f450,1994,3500,2585.0
355,chrysler,300m,1999,3300,2202.0
380,pontiac,firebird trans am ws6,1998,29950,2104.0


In [9]:
# 3.5 — Filtrer les outliers et retirer les colonnes utilitaires
n_before = len(df)
df = df[~df["_is_outlier_msrp"]].drop(columns=["_is_outlier_msrp", "_make_norm", "msrp_max"])
print(f"Après filtrage MSRP : {len(df):,} lignes ({n_before - len(df):,} retirées)")

Après filtrage MSRP : 354,067 lignes (8,125 retirées)


## 4. Gestion des valeurs manquantes

Stratégie par colonne :

| Colonne | Stratégie | Justification |
|---|---|---|
| `odometer` | Imputation par la médiane | Variable numérique très importante, ~1 % de NaN |
| `manufacturer`, `year` | Drop (déjà fait) | Indispensables pour la modélisation |
| `model` | Catégorie `"unknown"` | Cardinalité énorme, on évite de pousser sur le mode |
| `condition`, `cylinders`, `size`, `drive`, `type`, `fuel`, `transmission`, `paint_color`, `title_status` | Catégorie `"unknown"` | Préserve l'information "non renseigné", évite de fausser les distributions |
| `lat`, `long` | Imputation par la médiane par `state` | Géographique : préserver la cohérence régionale |
| `posting_date` | Imputation par la date médiane | Faible volume de NaN |
| `region`, `state` | Catégorie `"unknown"` | Conservés pour analyse géographique |

In [10]:
# 4.1 — Numérique : odometer
df["odometer"] = df["odometer"].fillna(df["odometer"].median())

# 4.2 — Catégoriel : remplacer NaN par "unknown"
CAT_COLS = [
    "model", "condition", "cylinders", "fuel", "title_status",
    "transmission", "drive", "size", "type", "paint_color",
    "state", "region",
]
for col in CAT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("unknown").astype(str).str.lower().str.strip()

# 4.3 — Géo : médiane par state pour lat/long
for col in ["lat", "long"]:
    if col in df.columns:
        df[col] = df.groupby("state")[col].transform(lambda s: s.fillna(s.median()))
        df[col] = df[col].fillna(df[col].median())  # fallback si tout NaN dans un state

# 4.4 — posting_date
df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce", utc=True)
df["posting_date"] = df["posting_date"].fillna(df["posting_date"].median())

print("NaN restants par colonne :")
df.isna().sum().sort_values(ascending=False).head(10)

NaN restants par colonne :


county         354067
region              0
price               0
long                0
lat                 0
state               0
paint_color         0
type                0
size                0
drive               0
dtype: int64

## 5. Feature engineering

On enrichit le dataset avec des features dérivées qui capturent des signaux importants pour la prédiction du prix :

| Catégorie | Features créées | Intuition métier |
|---|---|---|
| **Temporel** | `car_age`, `miles_per_year`, `posting_month`, `posting_year` | Âge, intensité d'usage, saisonnalité du marché |
| **Texte** (extrait de `description`) | `is_first_owner`, `has_service_history`, `desc_length` | Une 1ʳᵉ main + un historique d'entretien rassurent et tirent le prix vers le haut |
| **Géographique** | `is_rust_belt` | Les voitures du Rust Belt (MI, OH, IN, IL, PA, NY, WV, WI) subissent la neige et le sel routier → corrosion → décote |
| **Marque** | `is_premium_brand` | Effet additif fort sur le prix indépendamment du modèle |

In [11]:
# 5.1 — Features temporelles
df["car_age"] = (SCRAPE_YEAR - df["year"]).astype("int16")
df["posting_month"] = df["posting_date"].dt.month.astype("int8")
df["posting_year"] = df["posting_date"].dt.year.astype("int16")
df = df.drop(columns=["posting_date"])  # remplacée par les features dérivées

# Kilométrage annuel moyen : 100k miles sur 2 ans (VTC) != 100k miles sur 20 ans (garage)
df["miles_per_year"] = df["odometer"] / df["car_age"].clip(lower=1)

# 5.2 — Features texte extraites de description (regex case-insensitive)
desc = df["description"].fillna("").astype(str).str.lower()

FIRST_OWNER_PATTERN = r"\b(1st owner|one owner|first owner|single owner|original owner)\b"
df["is_first_owner"] = desc.str.contains(FIRST_OWNER_PATTERN, regex=True, na=False).astype("int8")

SERVICE_PATTERN = (
    r"\b(service records|service history|dealer maintained|maintenance records|"
    r"recent service|recently serviced|new tires|new brakes|just serviced|"
    r"all maintenance|service receipts)\b"
)
df["has_service_history"] = desc.str.contains(SERVICE_PATTERN, regex=True, na=False).astype("int8")

# Longueur de l'annonce : un vendeur qui détaille tend à valoriser/justifier un meilleur prix
df["desc_length"] = desc.str.split().str.len().fillna(0).astype("int32")
df = df.drop(columns=["description"])  # plus utile, déjà exploitée

# 5.3 — Features géographiques : Rust Belt
RUST_BELT_STATES = {"mi", "oh", "in", "il", "pa", "ny", "wv", "wi"}
df["is_rust_belt"] = df["state"].str.lower().isin(RUST_BELT_STATES).astype("int8")

# 5.4 — Marque premium
PREMIUM_BRANDS = {
    "bmw", "mercedes-benz", "audi", "lexus", "porsche",
    "acura", "infiniti", "jaguar", "land rover", "tesla",
    "cadillac", "lincoln", "genesis", "alfa-romeo",
}
df["is_premium_brand"] = df["manufacturer"].str.lower().isin(PREMIUM_BRANDS).astype("int8")

# Diagnostics rapides
new_features = [
    "car_age", "miles_per_year", "posting_month", "posting_year",
    "is_first_owner", "has_service_history", "desc_length",
    "is_rust_belt", "is_premium_brand",
]
print(f"Shape après feature engineering : {df.shape}")
print("\nTaux d'activation des flags binaires :")
for col in ["is_first_owner", "has_service_history", "is_rust_belt", "is_premium_brand"]:
    print(f"  {col:<22} : {100*df[col].mean():.2f} %")
df[new_features].head()

Shape après feature engineering : (354067, 24)


,age,posting_month,posting_year,is_premium_brand,odometer_per_year
0,7,5,2021,0,8274.714286
1,11,5,2021,0,6475.363636
2,1,5,2021,0,19160.000000
3,4,5,2021,0,10281.000000
4,8,5,2021,0,16000.000000


## 6. Train/test split

On splitte **avant** d'encoder les variables catégorielles, pour éviter toute fuite d'information du test dans l'encodage (notamment pour le target encoding sur `manufacturer`).

In [12]:
TARGET = "price"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train : {X_train.shape[0]:,} lignes")
print(f"Test  : {X_test.shape[0]:,} lignes")

Train : 283,253 lignes
Test  : 70,814 lignes


## 7. Encoding des variables catégorielles

| Type d'encodage | Colonnes | Justification |
|---|---|---|
| **Target encoding** (smooth mean) | `manufacturer` | ~40 modalités, fort signal sur le prix → encodage informatif |
| **Frequency encoding** | `model`, `region` | Très haute cardinalité (10k+ pour `model`) → one-hot ferait exploser la dim |
| **One-hot encoding** | `condition`, `fuel`, `transmission`, `drive`, `size`, `type`, `paint_color`, `cylinders`, `title_status`, `state` | Cardinalité faible/moyenne (≤ 50) → one-hot reste raisonnable |

Tous les encoders sont **fit sur le train uniquement**, puis appliqués au test.

In [13]:
# 7.1 — Target encoding sur manufacturer (mean target par marque, lissé)
def smooth_target_encode(
    train_x: pd.Series, train_y: pd.Series, test_x: pd.Series, smoothing: int = 20
) -> tuple[pd.Series, pd.Series]:
    """Target encoding lissé : mélange entre la moyenne globale et la moyenne par modalité."""
    global_mean = train_y.mean()
    stats = train_y.groupby(train_x).agg(["mean", "count"])
    smoothed = (
        stats["mean"] * stats["count"] + global_mean * smoothing
    ) / (stats["count"] + smoothing)
    return train_x.map(smoothed).fillna(global_mean), test_x.map(smoothed).fillna(global_mean)

X_train["manufacturer_te"], X_test["manufacturer_te"] = smooth_target_encode(
    X_train["manufacturer"], y_train, X_test["manufacturer"]
)
X_train = X_train.drop(columns=["manufacturer"])
X_test = X_test.drop(columns=["manufacturer"])

In [14]:
# 7.2 — Frequency encoding sur model et region
for col in ["model", "region"]:
    freq = X_train[col].value_counts(normalize=True)
    X_train[f"{col}_freq"] = X_train[col].map(freq).fillna(0.0)
    X_test[f"{col}_freq"] = X_test[col].map(freq).fillna(0.0)
    X_train = X_train.drop(columns=[col])
    X_test = X_test.drop(columns=[col])

In [15]:
# 7.3 — One-hot encoding sur les autres catégorielles (cardinalité faible/moyenne)
OHE_COLS = [
    "condition", "fuel", "transmission", "drive", "size",
    "type", "paint_color", "cylinders", "title_status", "state",
]
OHE_COLS = [c for c in OHE_COLS if c in X_train.columns]

X_train = pd.get_dummies(X_train, columns=OHE_COLS, drop_first=True, dtype="int8")
X_test = pd.get_dummies(X_test, columns=OHE_COLS, drop_first=True, dtype="int8")

# Aligner les colonnes train/test (au cas où une modalité n'apparaîtrait pas dans le test)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print(f"Shape après one-hot : train={X_train.shape}, test={X_test.shape}")

Shape après one-hot : train=(283253, 123), test=(70814, 123)


## 8. Scaling des variables numériques

On scale uniquement les colonnes **réellement numériques continues** (`year`, `age`, `odometer`, `lat`, `long`, `manufacturer_te`, `model_freq`, `region_freq`, `odometer_per_year`, `posting_year`, `posting_month`).

Les colonnes one-hot (déjà dans `[0, 1]`) et `is_premium_brand` (binaire) ne sont pas scalées.

**Pour les modèles linéaires** : indispensable pour que les coefficients soient comparables et que la régularisation fonctionne.
**Pour les modèles d'arbres** (RF, XGBoost, LightGBM) : sans impact, mais ne pose pas de problème non plus.

In [16]:
# Variables réellement numériques continues à scaler
NUM_COLS = [
    "year", "car_age", "odometer", "lat", "long",
    "manufacturer_te", "model_freq", "region_freq",
    "miles_per_year", "posting_year", "posting_month",
    "desc_length",
]
NUM_COLS = [c for c in NUM_COLS if c in X_train.columns]

scaler = StandardScaler()
X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

print("Stats après scaling (train) :")
X_train[NUM_COLS].describe().round(2).T[["mean", "std", "min", "max"]]

Stats après scaling (train) :


,mean,std,min,max
year,0.0,1.0,-5.42,1.60
age,0.0,1.0,-1.60,5.42
odometer,0.0,1.0,-1.47,6.48
lat,0.0,1.0,-20.58,7.21
long,-0.0,1.0,-3.62,14.82
manufacturer_te,-0.0,1.0,-2.80,13.40
model_freq,0.0,1.0,-0.62,4.84
region_freq,-0.0,1.0,-2.02,1.64
odometer_per_year,-0.0,1.0,-1.66,39.99
posting_year,0.0,0.0,0.00,0.00


## 9. Sauvegarde du dataset processé

On sauvegarde un seul fichier parquet contenant `X` + `y` + une colonne `_split`. `src/data.py::load_dataset_split()` lira ce fichier.

In [17]:
train_out = X_train.copy()
train_out[TARGET] = y_train
train_out["_split"] = "train"

test_out = X_test.copy()
test_out[TARGET] = y_test
test_out["_split"] = "test"

processed = pd.concat([train_out, test_out], axis=0, ignore_index=True)

# Sécurité : tout caster en numérique (les bool issus du get_dummies → int8)
for col in processed.columns:
    if processed[col].dtype == bool:
        processed[col] = processed[col].astype("int8")

processed.to_parquet(OUTPUT_PATH, index=False)
print(f"Sauvegardé : {OUTPUT_PATH}")
print(f"Shape       : {processed.shape}")
print(f"Colonnes    : {list(processed.columns)[:15]} ...")

Sauvegardé : ../data/vehicles_processed.parquet
Shape       : (354067, 125)
Colonnes    : ['year', 'odometer', 'county', 'lat', 'long', 'age', 'posting_month', 'posting_year', 'is_premium_brand', 'odometer_per_year', 'manufacturer_te', 'model_freq', 'region_freq', 'condition_fair', 'condition_good'] ...


## 10. Vérification : chargement via `src/data.py`

On valide que le contrat de `load_dataset_split()` est respecté.

In [18]:
import sys
sys.path.insert(0, str(Path("..") / "src"))

from data import load_dataset_split  # noqa: E402

X_train_l, X_test_l, y_train_l, y_test_l = load_dataset_split()
print(f"X_train : {X_train_l.shape}")
print(f"X_test  : {X_test_l.shape}")
print(f"y_train : {y_train_l.shape} — {y_train_l.dtype}")
print(f"y_test  : {y_test_l.shape} — {y_test_l.dtype}")
assert X_train_l.shape[1] == X_test_l.shape[1], "Mismatch de colonnes train/test"
assert len(X_train_l) == len(y_train_l) and len(X_test_l) == len(y_test_l)
print("\n✓ Contrat load_dataset_split() respecté.")

X_train : (283253, 123)
X_test  : (70814, 123)
y_train : (283253,) — int64
y_test  : (70814,) — int64

✓ Contrat load_dataset_split() respecté.
